In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import cohen_kappa_score, log_loss, brier_score_loss
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAS_LGBM = True
except: HAS_LGBM = False

# ===================================================
# CONFIG - EXACT V44
# ===================================================
ID_COL = "trustii_id"
AGE_COL = "Age_Baseline"
SEX_COL = "Genre"
TARGET_SEV = "OUTCOME SEVERITY"
TARGET_MACE = "OUTCOME MACE"
SNP_COLS = [f"SNP{i}" for i in range(1, 289)]
PRIORITY_SNP = [f"SNP{i}" for i in range(1, 76)]
DATA_DIR = "/C:/Users/boomi/Documents/Boomika_Files/M1_GENIOMHE/P1/Data_analysis and deep learning/LMDM-m1geniomhe-group-8/code"
SEV_WEIGHTS = {0: 1.5, 1: 1}

# ===================================================
# LOAD & PREPARE
# ===================================================
print("=" * 60)
print("V51 - V44 MACE + IMPROVED SEVERITY")
print("=" * 60)

train = pd.read_csv(f"{DATA_DIR}/cardihack_final-train.csv")
test = pd.read_csv(f"{DATA_DIR}/cardihack_final-test.csv")
dummy = pd.read_csv(f"{DATA_DIR}/cardihack_dummy_pred.csv")

def filter_maf(df, snps, thresh=0.005):
    out = []
    for s in snps:
        v = df[s].dropna()
        if len(v) < 10: continue
        f = v.value_counts()
        if len(f) < 2: continue
        if f.min() / f.sum() >= thresh: out.append(s)
    return out

def chi2_snps(df, target, snps, p_thresh=0.1):
    pvals = []
    for s in snps:
        tab = pd.crosstab(df[s].fillna(-1), df[target])
        if tab.shape[0] < 2 or tab.shape[1] < 2: continue
        try:
            _, p, _, _ = chi2_contingency(tab)
            pvals.append((s, p))
        except: pass
    pvals.sort(key=lambda x: x[1])
    return [s for s, p in pvals if p < p_thresh]

def enforce_priority(selected, priority, max_total=120):
    pin = [s for s in selected if s in priority]
    other = [s for s in selected if s not in priority]
    minp = max(5, int(max_total * 0.2))
    if len(pin) < minp:
        add = [s for s in priority if s not in pin]
        pin.extend(add[:minp - len(pin)])
    return (pin + other)[:max_total]

def engineer(df):
    df = df.copy()
    if AGE_COL in df.columns:
        df['Age_bin'] = pd.cut(df[AGE_COL], bins=[0,50,60,70,100], labels=[0,1,2,3])
        df['Age_sq'] = df[AGE_COL] ** 2
    if SEX_COL in df.columns:
        df[SEX_COL] = df[SEX_COL].map({'M':1,'F':0}).fillna(0.5)
    return df

def isotonic_cal(oof, test_p, y):
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(oof, y)
    return np.clip(iso.predict(test_p), 0.01, 0.99)

def weighted_ll(y, p, w=SEV_WEIGHTS):
    sw = np.array([w[y_] for y_ in y])
    return log_loss(y, p, sample_weight=sw)

train = engineer(train)
test = engineer(test)

clinical = [c for c in train.columns
            if c not in SNP_COLS + [ID_COL, TARGET_MACE, TARGET_SEV]
            and c in test.columns and train[c].dtype in ['float64','int64']]

y_mace = train[TARGET_MACE].values
y_sev = train[TARGET_SEV].values
base_rate = y_sev.mean()

# ===================================================
# V44 EXACT MACE (DO NOT CHANGE!)
# ===================================================
print("\n" + "=" * 60)
print("V44 EXACT MACE (PROVEN: 0.3116)")
print("=" * 60)

mace_snps = filter_maf(train, SNP_COLS, 0.005)
mace_snps = chi2_snps(train, TARGET_MACE, mace_snps, 0.10)
mace_snps = enforce_priority(mace_snps, PRIORITY_SNP, 120)

mace_feats = [AGE_COL, SEX_COL, 'Age_bin', 'Age_sq'] + clinical + mace_snps
mace_feats = [f for f in mace_feats if f in train.columns and f in test.columns]
mace_feats = list(dict.fromkeys(mace_feats))

mace_model = Pipeline([
    ("imp", IterativeImputer(max_iter=20, random_state=42, imputation_order='random')),
    ("sc", RobustScaler()),
    ("clf", LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.3, C=0.3,
        max_iter=10000, multi_class="multinomial",
        class_weight='balanced', random_state=42
    ))
])
mace_model.fit(train[mace_feats], y_mace)
test_mace_proba = mace_model.predict_proba(test[mace_feats])

# V44 EXACT THRESHOLDS
t1, t2 = 0.35, 0.50
mace_pred = np.zeros(len(test), dtype=int)
mace_pred[test_mace_proba[:, 2] > t2] = 2
mace_pred[(test_mace_proba[:, 1] > t1) & (test_mace_proba[:, 2] <= t2)] = 1

print(f"MACE: {np.bincount(mace_pred, minlength=3)} (V44 exact)")

# ===================================================
# IMPROVED SEVERITY MODELS
# ===================================================
print("\n" + "=" * 60)
print("IMPROVED SEVERITY (Exploring variations)")
print("=" * 60)

sev_snps = filter_maf(train, SNP_COLS, 0.005)
sev_snps = chi2_snps(train, TARGET_SEV, sev_snps, 0.15)

sev_clinical = [AGE_COL, SEX_COL, 'Age_bin', 'Age_sq'] + clinical
sev_clinical = [f for f in sev_clinical if f in train.columns and f in test.columns]

sev_full = sev_clinical + sev_snps[:50]
sev_full = [f for f in sev_full if f in train.columns and f in test.columns]

# Impute
imp_c = IterativeImputer(max_iter=20, random_state=42)
X_tr_c = imp_c.fit_transform(train[sev_clinical])
X_te_c = imp_c.transform(test[sev_clinical])

imp_f = IterativeImputer(max_iter=20, random_state=42)
X_tr_f = imp_f.fit_transform(train[sev_full])
X_te_f = imp_f.transform(test[sev_full])

sev_oof = {}
sev_test = {}

# V44 Models
print("\n   Training V44 severity models...")
m1 = Pipeline([("sc", RobustScaler()), ("clf", LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.3, C=0.5, max_iter=10000, class_weight='balanced', random_state=42))])
m1.fit(X_tr_c, y_sev); sev_oof['lr1'] = cross_val_predict(m1, X_tr_c, y_sev, cv=5, method='predict_proba')[:, 1]; sev_test['lr1'] = m1.predict_proba(X_te_c)[:, 1]

m2 = Pipeline([("sc", RobustScaler()), ("clf", LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.3, C=0.3, max_iter=10000, class_weight='balanced', random_state=42))])
m2.fit(X_tr_c, y_sev); sev_oof['lr2'] = cross_val_predict(m2, X_tr_c, y_sev, cv=5, method='predict_proba')[:, 1]; sev_test['lr2'] = m2.predict_proba(X_te_c)[:, 1]

m3 = Pipeline([("sc", RobustScaler()), ("clf", LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.3, C=0.5, max_iter=10000, class_weight='balanced', random_state=42))])
m3.fit(X_tr_f, y_sev); sev_oof['lr3'] = cross_val_predict(m3, X_tr_f, y_sev, cv=5, method='predict_proba')[:, 1]; sev_test['lr3'] = m3.predict_proba(X_te_f)[:, 1]

m4 = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, subsample=0.8, random_state=42)
m4.fit(X_tr_c, y_sev); sev_oof['gb1'] = cross_val_predict(m4, X_tr_c, y_sev, cv=5, method='predict_proba')[:, 1]; sev_test['gb1'] = m4.predict_proba(X_te_c)[:, 1]

m5 = GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate=0.05, subsample=0.7, random_state=42)
m5.fit(X_tr_c, y_sev); sev_oof['gb2'] = cross_val_predict(m5, X_tr_c, y_sev, cv=5, method='predict_proba')[:, 1]; sev_test['gb2'] = m5.predict_proba(X_te_c)[:, 1]

m6 = RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=10, class_weight='balanced', random_state=42, n_jobs=-1)
m6.fit(X_tr_c, y_sev); sev_oof['rf'] = cross_val_predict(m6, X_tr_c, y_sev, cv=5, method='predict_proba')[:, 1]; sev_test['rf'] = m6.predict_proba(X_te_c)[:, 1]

m7 = Pipeline([("sc", RobustScaler()), ("clf", LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, C=0.1, max_iter=10000, class_weight='balanced', random_state=42))])
m7.fit(X_tr_c, y_sev); sev_oof['lr4'] = cross_val_predict(m7, X_tr_c, y_sev, cv=5, method='predict_proba')[:, 1]; sev_test['lr4'] = m7.predict_proba(X_te_c)[:, 1]

# LightGBM if available
if HAS_LGBM:
    m8 = lgb.LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, class_weight='balanced', random_state=42, verbose=-1)
    m8.fit(X_tr_c, y_sev); sev_oof['lgb'] = cross_val_predict(m8, X_tr_c, y_sev, cv=5, method='predict_proba')[:, 1]; sev_test['lgb'] = m8.predict_proba(X_te_c)[:, 1]

# Brier-weighted ensemble
brier = {k: brier_score_loss(y_sev, v) for k, v in sev_oof.items()}
inv_b = {k: 1/v for k, v in brier.items()}
total = sum(inv_b.values())
weights = {k: v/total for k, v in inv_b.items()}

print("\nSeverity Brier scores:")
for k in sorted(brier, key=brier.get):
    print(f"   {k}: {brier[k]:.4f}")

# Ensembles
oof_v44 = sum(sev_oof[k] * weights[k] for k in weights)
test_v44 = sum(sev_test[k] * weights[k] for k in weights)

# V44 isotonic
sev_v44_iso = isotonic_cal(oof_v44, test_v44, y_sev)

# Simple calibration (scale to base rate)
cal_factor = base_rate / test_v44.mean()
sev_v44_simple = np.clip(test_v44 * cal_factor, 0.01, 0.99)

# Weighted log loss comparison
wll_v44 = weighted_ll(y_sev, oof_v44)
print(f"\nV44 OOF Weighted LL: {wll_v44:.4f}")
print(f"V44 isotonic mean: {sev_v44_iso.mean():.3f}")
print(f"V44 simple mean: {sev_v44_simple.mean():.3f}")
print(f"Base rate: {base_rate:.3f}")

# ===================================================
# GENERATE SUBMISSIONS
# ===================================================
print("\n" + "=" * 60)
print("SUBMISSIONS")
print("=" * 60)

# V44 exact (isotonic) - should be 0.3116
sub1 = pd.DataFrame({ID_COL: test[ID_COL], TARGET_MACE: mace_pred, TARGET_SEV: sev_v44_iso})
sub1 = sub1[dummy.columns]
sub1.to_csv(f"{DATA_DIR}/v51_v44_exact.csv", index=False)

# V44 with simple calibration instead
sub2 = pd.DataFrame({ID_COL: test[ID_COL], TARGET_MACE: mace_pred, TARGET_SEV: sev_v44_simple})
sub2 = sub2[dummy.columns]
sub2.to_csv(f"{DATA_DIR}/v51_simple_cal.csv", index=False)

print(f"\n   v51_v44_exact.csv: MACE {np.bincount(mace_pred, minlength=3)}, Sev mean={sev_v44_iso.mean():.3f}")
print(f"   v51_simple_cal.csv: MACE {np.bincount(mace_pred, minlength=3)}, Sev mean={sev_v44_simple.mean():.3f}")

print("\n✅ Test v51_v44_exact.csv first (score match 0.3116)")